<a href="https://colab.research.google.com/github/AleynaCifcibasi/Python-ile-Machine-Learning-al-malar-ve-projeleri/blob/main/Python_Oturum_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Faster R-CNN


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/Tech İstanbul"

Advertising.csv		Python-Oturum-1.ipynb  Python-Oturum-6.ipynb
cnn.keras		Python-Oturum-2.ipynb  Python-Oturum-7.ipynb
memekanseri.data	Python-Oturum-3.ipynb  Python-Oturum-8.ipynb
phishing_email.csv	Python-Oturum-4.ipynb  Python-Oturum-9.ipynb
Python-Oturum-10.ipynb	Python-Oturum-5.ipynb  rnn.h5


In [ ]:
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-nx_vcb7h
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-nx_vcb7h
  Resolved https://github.com/facebookresearch/detectron2.git to commit 1e3e13bbf607b54f62205c4c33922521822fb298
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.9 MB/s eta 0:00:00
  Created wheel for 

In [ ]:
import torch
import torchvision
import cv2
import os
import numpy as np
import json
import random
import matplotlib.pyplot as plt
from detectron2.structures import BoxMode
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import ColorMode, Visualizer
from matplotlib.patches import Rectangle

/usr/local/lib/python3.13/dist-packages/detectron2/model_zoo/model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
def get_data_dicts(directory, classes):
  dataset_dicts = []
  for filename in [file for file in os.listdir(directory) if file.endswith('.json')]:
    json_file = os.path.join(directory,filename)
    with open(json_file) as f:
      img_anns = json.load(f)

    record = {}
    filename = os.path.join(directory, img_anns['imagePath'])

    record['file_name'] = filename
    record['height'] = 700
    record['width'] = 700

    annos = img_anns['shapes']
    objs = []

    for anno in annos:
      px = [a[0] for a in anno['points']]
      py = [a[1] for a in anno['points']]

      poly = [(x,y) for x,y in zip(px,py)]
      poly = [p for x in poly for p in x]

      obj = {
          'bbox': [np.min(px), np.min(py), np.max(px), np.max(py)],
          'bbox_mode': BoxMode.XYXY_ABS,
          'category_id': classes.index(anno['label']),
          'iscrowd': 0
      }

      objs.append(obj)
      record['annotations'] = objs
      dataset_dicts.append(record)
  return dataset_dicts


In [ ]:
classes = ['kedi','kopek']

data_path = '/content/drive/MyDrive/Tech İstanbul/veriseti/'

for d in ['train','test']:
  DatasetCatalog.register(
      'my_'+d,
      lambda d=d: get_data_dicts(data_path+d,classes)
  )

  MetadataCatalog.get('my_'+d).set(thing_classes=classes)

microcontroller_metadata = MetadataCatalog.get('my_train')

##Config

In [ ]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file('COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml'))
cfg.DATASETS.TRAIN = ('my_train',)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url('COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml')
cfg.SOLVER.IMS_PER_BATCH = 1
cfg.SOLVER.BASE_LR = 0.001
cfg.SOLVER.STEPS = [500]
cfg.SOLVER.GAMMA = 0.05
cfg.TEST.EVAL_PERIOD = 100
cfg.SOLVER.MAX_ITER = 2000
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2

In [ ]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
egitim = DefaultTrainer(cfg)
egitim.resume_or_load(resume=False)

[10/12 13:48:35 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Tech İstanbul/veriseti/train'

In [ ]:
egitim.train()

NameError: name 'egitim' is not defined

In [ ]:
##TEST

In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR,'model_final.pth')
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.6
cfg.DATASETS.TEST = ('my_test',)
tahmin = DefaultPredictor(cfg)

test_metadata = MetadataCatalog.get('my_test')
test_dataset_dicts = get_data_dicts(data_path + 'test',classes)

[10/12 13:55:24 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output/model_final.pth ...


AssertionError: Checkpoint ./output/model_final.pth not found!

##Görselleştirme

In [ ]:
for a in random.sample(test_dataset_dicts,2):
  img = cv2.imread(a['file_name'])
  ciktilar = tahmin(img)
  g = Visualizer(img[:,:,::-1],
                 metadata=microcontroller_metadata,
                 scale=0.8)
  g = g.draw_instance_predictions(ciktilar['instances'].to('cpu'))
  plt.figure(figsize=(20,10))
  plt.imshow(cv2.cvtColor(g.get_image()[:,:,::-1], cv2.COLOR_BGR2RGB))
  plt.show()

NameError: name 'test_dataset_dicts' is not defined